# modeldelta — Phase 1: Instruct Controls + OLMo Stage Chains

**Goal:** Fill in the remaining pairs for Phase 1 of the geometric regimes experiment.

**Pairs to compute:**
- **B1** `allenai/OLMo-2-1124-7B` → `-SFT` (base_to_sft)
- **B2** `allenai/OLMo-2-1124-7B-SFT` → `-DPO` (sft_to_dpo)
- **B3** `allenai/OLMo-2-1124-7B-DPO` → `-Instruct` (dpo_to_instruct)
- **B4** `allenai/OLMo-2-1124-13B` → `-SFT` (base_to_sft)
- **B5** `allenai/OLMo-2-1124-13B-SFT` → `-DPO` (sft_to_dpo)
- **B6** `allenai/OLMo-2-1124-13B-DPO` → `-Instruct` (dpo_to_instruct)
- **A10** `google/gemma-2-27b` → `google/gemma-2-27b-it` (instruct)
- **A5** `Qwen/Qwen2.5-32B` → `Qwen/Qwen2.5-32B-Instruct` (instruct)

**Runtime:** CPU high-RAM (51 GB, 0.24 units/hr)  
**Disk:** 250 GB — models cleared between groups  
**Estimated time:** ~3–4 hours total  
**Output:** JSON saved to Drive + pushed to HF gallery

In [ ]:
!pip install modeldelta -q

In [ ]:
import os, gc, shutil, json, time
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN or ''

DRIVE_DIR = Path('/content/drive/MyDrive/modeldelta_reports')
DRIVE_DIR.mkdir(exist_ok=True)

print('Drive ready:', DRIVE_DIR)

# Check disk space
import shutil as _sh
total, used, free = _sh.disk_usage('/')
print(f'Disk: {free/1e9:.0f} GB free / {total/1e9:.0f} GB total')

In [ ]:
# ── Helper: run one pair, save to Drive, push to HF ──────────

def run_pair(model_a, model_b, pair_id, label, skip_if_exists=True):
    """Run modeldelta on a pair. Returns results dict or None if cached."""
    from modeldelta.utils.model_loader import resolve_model, get_tensor_map
    from modeldelta.utils.model_meta import validate_pair, fetch_model_meta
    from modeldelta.core.weight_diff import compare_models
    from modeldelta.report.json_report import generate_json
    from modeldelta.database.hub import push_results

    out_path = DRIVE_DIR / f'{pair_id}.json'
    if skip_if_exists and out_path.exists():
        print(f'  SKIP {pair_id}: already on Drive')
        return None

    print(f'\n{"="*60}')
    print(f'{pair_id} ({label})')
    print(f'{model_a}')
    print(f'  → {model_b}')
    print(f'{"="*60}')

    t0 = time.time()

    meta_a, meta_b, warnings = validate_pair(model_a, model_b, token=HF_TOKEN)
    for w in warnings:
        print(f'Warning: {w}')

    print('Downloading models...')
    path_a = resolve_model(model_a, token=HF_TOKEN, quiet=True)
    path_b = resolve_model(model_b, token=HF_TOKEN, quiet=True)
    print(f'  Downloaded in {time.time()-t0:.0f}s')

    tm_a = get_tensor_map(path_a)
    tm_b = get_tensor_map(path_b)

    def progress(i, total, elapsed, name):
        if i % 100 == 0 or i == total:
            print(f'  [{i}/{total}] {elapsed:.0f}s — {name}')

    results, n_skipped = compare_models(tm_a, tm_b, progress_callback=progress)
    t_elapsed = time.time() - t0
    print(f'Done: {len(results)} tensors, {n_skipped} skipped, {t_elapsed:.0f}s')

    # Save JSON to Drive
    json_str = generate_json(results, model_a, model_b, n_skipped,
                              meta_a=meta_a, meta_b=meta_b)
    with open(out_path, 'w') as f:
        f.write(json_str)
    print(f'Saved: {out_path}')

    # Push to HF gallery
    try:
        pid = push_results(results, model_a, model_b, n_skipped,
                           meta_a=meta_a, meta_b=meta_b,
                           token=HF_TOKEN, computed_by='colab')
        print(f'Gallery: pair_id={pid}')
    except Exception as e:
        print(f'Gallery push failed: {e}')

    return results


def clear_cache(*model_ids):
    """Delete HF cache for given model IDs."""
    cache_dir = Path.home() / '.cache' / 'huggingface' / 'hub'
    freed = 0
    for mid in model_ids:
        d = cache_dir / f'models--{mid.replace("/", "--")}'
        if d.exists():
            size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file())
            shutil.rmtree(d)
            freed += size
            print(f'  Cleared: {mid} ({size/1e9:.1f} GB)')
    if freed:
        print(f'  Total freed: {freed/1e9:.1f} GB')
    gc.collect()


def disk_free():
    total, used, free = shutil.disk_usage('/')
    print(f'Disk: {free/1e9:.0f} GB free')


print('Helpers ready')

---
## Bucket B — OLMo-2 7B Stage Chain

4 checkpoints (~14 GB each), 3 consecutive pairs.  
Keep all 4 in cache while running the chain, then clear.

In [ ]:
# Pre-download all 4 OLMo-2-7B checkpoints so pairs share cache
from modeldelta.utils.model_loader import resolve_model

OLMO7_MODELS = [
    'allenai/OLMo-2-1124-7B',
    'allenai/OLMo-2-1124-7B-SFT',
    'allenai/OLMo-2-1124-7B-DPO',
    'allenai/OLMo-2-1124-7B-Instruct',
]

disk_free()
print('Downloading OLMo-2 7B checkpoints...')
t0 = time.time()
for mid in OLMO7_MODELS:
    print(f'  {mid}...')
    resolve_model(mid, token=HF_TOKEN, quiet=True)
print(f'All 4 checkpoints ready in {time.time()-t0:.0f}s')
disk_free()

In [ ]:
# B1: Base → SFT
run_pair(
    'allenai/OLMo-2-1124-7B',
    'allenai/OLMo-2-1124-7B-SFT',
    'B1_olmo2_7b_base_to_sft',
    'base_to_sft',
)

In [ ]:
# B2: SFT → DPO
run_pair(
    'allenai/OLMo-2-1124-7B-SFT',
    'allenai/OLMo-2-1124-7B-DPO',
    'B2_olmo2_7b_sft_to_dpo',
    'sft_to_dpo',
)

In [ ]:
# B3: DPO → Instruct
run_pair(
    'allenai/OLMo-2-1124-7B-DPO',
    'allenai/OLMo-2-1124-7B-Instruct',
    'B3_olmo2_7b_dpo_to_instruct',
    'dpo_to_instruct',
)

# Free disk
clear_cache(*OLMO7_MODELS)
disk_free()

---
## Bucket B — OLMo-2 13B Stage Chain

4 checkpoints (~26 GB each = ~104 GB total).  
Keep all 4 in cache, run 3 pairs, then clear.

In [ ]:
OLMO13_MODELS = [
    'allenai/OLMo-2-1124-13B',
    'allenai/OLMo-2-1124-13B-SFT',
    'allenai/OLMo-2-1124-13B-DPO',
    'allenai/OLMo-2-1124-13B-Instruct',
]

disk_free()
print('Downloading OLMo-2 13B checkpoints...')
t0 = time.time()
for mid in OLMO13_MODELS:
    print(f'  {mid}...')
    resolve_model(mid, token=HF_TOKEN, quiet=True)
print(f'All 4 checkpoints ready in {time.time()-t0:.0f}s')
disk_free()

In [ ]:
# B4: Base → SFT
run_pair(
    'allenai/OLMo-2-1124-13B',
    'allenai/OLMo-2-1124-13B-SFT',
    'B4_olmo2_13b_base_to_sft',
    'base_to_sft',
)

In [ ]:
# B5: SFT → DPO
run_pair(
    'allenai/OLMo-2-1124-13B-SFT',
    'allenai/OLMo-2-1124-13B-DPO',
    'B5_olmo2_13b_sft_to_dpo',
    'sft_to_dpo',
)

In [ ]:
# B6: DPO → Instruct
run_pair(
    'allenai/OLMo-2-1124-13B-DPO',
    'allenai/OLMo-2-1124-13B-Instruct',
    'B6_olmo2_13b_dpo_to_instruct',
    'dpo_to_instruct',
)

clear_cache(*OLMO13_MODELS)
disk_free()

---
## A10 — Gemma-2-27B → Gemma-2-27B-it

~54 GB per checkpoint = ~108 GB total.

In [ ]:
disk_free()
run_pair(
    'google/gemma-2-27b',
    'google/gemma-2-27b-it',
    'A10_gemma2_27b_instruct',
    'instruct',
)
clear_cache('google/gemma-2-27b', 'google/gemma-2-27b-it')
disk_free()

---
## A5 — Qwen2.5-32B → Qwen2.5-32B-Instruct

~64 GB per checkpoint = ~128 GB total.

In [ ]:
disk_free()
run_pair(
    'Qwen/Qwen2.5-32B',
    'Qwen/Qwen2.5-32B-Instruct',
    'A5_qwen25_32b_instruct',
    'instruct',
)
clear_cache('Qwen/Qwen2.5-32B', 'Qwen/Qwen2.5-32B-Instruct')
disk_free()

---
## Summary

Check which pairs are now on Drive:

In [ ]:
expected = [
    'B1_olmo2_7b_base_to_sft',
    'B2_olmo2_7b_sft_to_dpo',
    'B3_olmo2_7b_dpo_to_instruct',
    'B4_olmo2_13b_base_to_sft',
    'B5_olmo2_13b_sft_to_dpo',
    'B6_olmo2_13b_dpo_to_instruct',
    'A10_gemma2_27b_instruct',
    'A5_qwen25_32b_instruct',
]

print('Drive status:')
for pid in expected:
    p = DRIVE_DIR / f'{pid}.json'
    status = f'OK ({p.stat().st_size/1e6:.0f} MB)' if p.exists() else 'MISSING'
    print(f'  {pid:<40} {status}')